<a href="https://colab.research.google.com/github/EAN001729/project_psd/blob/main/Project_PSD.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [2]:
import pandas as pd
import numpy as np

try:
    from google.colab import files
    uploaded = files.upload()  # pilih paket_a_transaksi_pos_2024_2025.csv
    nama_file = next(iter(uploaded))
except ImportError:

    nama_file = "paket_a_transaksi_pos_2024_2025.csv"

df_raw = pd.read_csv(nama_file, dtype=str, keep_default_na=False)
print("Ukuran data mentah:", df_raw.shape)
df_raw.head()

Saving paket_a_transaksi_pos_2024_2025.csv to paket_a_transaksi_pos_2024_2025 (1).csv
Ukuran data mentah: (5030, 9)


,id_transaksi,tanggal,kode_gerai,kode_kasir,kode_produk,jumlah,harga_satuan,diskon_persen,status
0,TRX0241065,2025-06-02,G09,K011,RMT1402,4,13000,0,SELESAI
1,TRX0240510,2024-12-28,G09,K015,SGR1205,1,21000,0,SELESAI
2,TRX0244483,2024-09-04,G11,K029,RMT1407,3,"7500,00",0,SELESAI
3,TRX0242532,2025-07-30,G05,K001,MKN1103,2,6500,0,SELESAI
4,TRX0243509,26/03/2025,G05,K016,RMT1407,1,6000,0,SELESAI


In [3]:

df = df_raw.apply(lambda col: col.str.strip()).replace("", np.nan)

missing_df = pd.DataFrame({
    "jumlah_missing": df.isna().sum(),
    "persen": (df.isna().mean() * 100).round(2)
})
missing_df[missing_df["jumlah_missing"] > 0]

,jumlah_missing,persen
kode_kasir,261,5.19


In [4]:
df["kode_kasir"] = df["kode_kasir"].fillna("TIDAK_DIKETAHUI")
df["diskon_persen"] = df["diskon_persen"].fillna("0")
df["status"] = df["status"].fillna("TIDAK_DIKETAHUI").str.upper()
df["kode_produk"] = df["kode_produk"].str.upper()

sebelum = len(df)
df = df.dropna(subset=["jumlah", "harga_satuan"])
print("Baris dibuang (jumlah/harga kosong):", sebelum - len(df))

Baris dibuang (jumlah/harga kosong): 0


In [5]:
# Cell 4
def bersihkan_angka(x):
    x = str(x).strip().replace(".", "").replace(",", ".")
    return float(x)

df["harga_satuan"] = df["harga_satuan"].apply(bersihkan_angka)
df["jumlah"] = pd.to_numeric(df["jumlah"], errors="coerce")
df["diskon_persen"] = pd.to_numeric(df["diskon_persen"], errors="coerce").fillna(0)

t1 = pd.to_datetime(df["tanggal"], format="%Y-%m-%d", errors="coerce")
t2 = pd.to_datetime(df["tanggal"], format="%d-%m-%Y", errors="coerce")
t3 = pd.to_datetime(df["tanggal"], format="%d/%m/%Y", errors="coerce")
df["tanggal"] = t1.fillna(t2).fillna(t3)
df = df.dropna(subset=["tanggal"])
df["tahun"] = df["tanggal"].dt.year.astype("Int64")

sebelum = len(df)
df = df.drop_duplicates(subset="id_transaksi")
print("Duplikat dibuang:", sebelum - len(df))
print("Sisa baris:", len(df))

Duplikat dibuang: 30
Sisa baris: 5000


In [6]:

df["subtotal"] = df["jumlah"] * df["harga_satuan"]
df["nilai_diskon"] = df["subtotal"] * df["diskon_persen"] / 100
df["pendapatan_bersih"] = df["subtotal"] - df["nilai_diskon"]
df["jumlah_wajar"] = df["jumlah"].between(1, 12)

print(df["status"].value_counts())
print("\nJumlah tidak wajar:", (~df["jumlah_wajar"]).sum())
print("Jumlah tidak wajar tapi SELESAI:",
      ((~df["jumlah_wajar"]) & (df["status"] == "SELESAI")).sum())

status
SELESAI    4719
BATAL       191
RETUR        90
Name: count, dtype: int64

Jumlah tidak wajar: 52
Jumlah tidak wajar tapi SELESAI: 47


In [7]:

data_hitung = df[
    (df["status"] == "SELESAI") &
    (df["jumlah_wajar"]) &
    (df["tahun"].isin([2024, 2025]))
]

pendapatan_tahunan = (
    data_hitung.groupby("tahun")
    .agg(
        jumlah_transaksi=("id_transaksi", "count"),
        total_item=("jumlah", "sum"),
        pendapatan_kotor=("subtotal", "sum"),
        total_diskon=("nilai_diskon", "sum"),
        pendapatan_bersih=("pendapatan_bersih", "sum"),
    )
    .round(0)
    .astype("int64")
    .reset_index()
)
pendapatan_tahunan

,tahun,jumlah_transaksi,total_item,pendapatan_kotor,total_diskon,pendapatan_bersih
0,2024,2327,5967,130883500,3477025,127406475
1,2025,2345,6050,138664000,3662900,135001100


In [8]:
# Cell 7
for _, row in pendapatan_tahunan.iterrows():
    kotor = f"{row['pendapatan_kotor']:,.0f}".replace(",", ".")
    bersih = f"{row['pendapatan_bersih']:,.0f}".replace(",", ".")
    print(f"Tahun {row['tahun']}: pendapatan kotor Rp {kotor} | setelah diskon Rp {bersih}")

Tahun 2024: pendapatan kotor Rp 130.883.500 | setelah diskon Rp 127.406.475
Tahun 2025: pendapatan kotor Rp 138.664.000 | setelah diskon Rp 135.001.100


In [19]:
pendapatan_tahunan.to_csv("hasil_pendapatan_per_tahun.csv", index=False)
print("File tersimpan: hasil_pendapatan_per_tahun.csv")

File tersimpan: hasil_pendapatan_per_tahun.csv


In [20]:
import os
import shutil
from google.colab import drive

drive.mount("/content/drive")

folder_drive = "/content/drive/MyDrive/hasil_pos"
os.makedirs(folder_drive, exist_ok=True)
shutil.copy("hasil_pendapatan_per_tahun.csv", folder_drive)

print("Isi folder Drive:", os.listdir(folder_drive))

Mounted at /content/drive
Isi folder Drive: ['hasil_pendapatan_per_tahun.csv']
